In [0]:
pip install rapidfuzz

Read the Silver data

In [0]:
# import libraries
from rapidfuzz import fuzz, process


silver_df = spark.read.table("cms_mdm_silver.cms_mdm_silver.ca_pediatrics_silver_data").toPandas()

display(silver_df)

Dataset overview

In [0]:
print(f"There are - {len(silver_df)} records in the silver DataFrame.")
print(f"total column counts is {len(silver_df.columns)} and column names are - {silver_df.columns}")
print(f"The data types are {silver_df.dtypes}")

Missing Value Analysis

In [0]:
import pandas as pd
print("===Missing Values===")

missing = (silver_df.isnull() | (silver_df.map(str).map(lambda x: x.strip() == ''))).sum()
missing_pct = (((silver_df.isnull() | (silver_df.map(str).map(lambda x: x.strip() == ''))).sum())/len(silver_df)*100).round(2)
missing_df = pd.concat([missing,missing_pct], axis=1)
missing_df = missing_df.rename(columns={0:'missing_cnt', 1:'missing_pct'})
missing_df = missing_df.sort_values(by='missing_cnt', ascending=False)
missing_df

Duplicate Detection 

In [0]:
print("===Duplicates===")
full_dupes = silver_df.duplicated().sum()
print(f"There are {full_dupes} full duplicates")

# duplicate Legacy_System_ID - that should be unique for each row
print("===Legacy_System_ID===")
legacy_dupes = silver_df['Legacy_System_ID'].duplicated().sum()
print(f"There are {legacy_dupes} Legacy_System_ID duplicates")

print("===NPI===")
npi_dupes = silver_df['NPI'].duplicated().sum()
print(f"There are {npi_dupes} NPI duplicates")

print("===Names===")
name_dupes = silver_df[['First_Name','Last_Name']].duplicated().sum()
print(f"There are {name_dupes} duplicate names")

print("===Phone Numbers===")
phone_dupes = silver_df['Phone'].duplicated().sum()
print(f"There are {phone_dupes} duplicate phone numbers")

print("===Address===")
address_dupes = silver_df[['Practice_Address_line_1','Practice_Address_line_2','City','State','Zip']].duplicated().sum()
print(f"There are {address_dupes} duplicate address")




Since Addresses and phone numbers have been standardized, only the NPI field has not been tested for consistency

In [0]:

# Check the number of NPIs that are not consistent - should be all 10 digits, not null, only contain numbers

print("$$$ NPI Consistency $$$")

npi = silver_df['NPI'].fillna('').astype(str)

NPI_not_consistent = (
    npi.str.contains(r'[^0-9]', regex=True)
    | (npi.str.len() != 10)
)

print(f"There are {NPI_not_consistent.sum()} NPIs that are not consistent")

print(f"There are {silver_df['NPI'].isnull().sum()} NPIs that are null")

# Inconsistent NPIs are all null, so no need to flag them as inconsistent



**Data Quality Scoring**

Score the silver dataset based on 2 MDM data quality dimensions:

- **Completeness** — are all critical fields populated?
- **Uniqueness** — are there duplicate records?


A composite Data Quality Score (DQS) will be calculated for each record.

In [0]:
# Define Critical MDM fields
critical_fields = ['NPI', 'First_Name', 'Last_Name','Practice_Address_line_1', 'City', 'State', 'Zip', 'Phone_std']


print("### COMPLETENESS SCORE ###")

# Score each row based on how many of the critical MDM fields are populated
silver_df['completeness_score'] = (((silver_df[critical_fields].notnull() & (silver_df[critical_fields].map(str).map(lambda x: x.strip() != ''))).sum(axis=1))/len(critical_fields)*100).round(2)

silver_df.head(5)


Uniqueness Scoring 

In [0]:
print("****Uniqueness Score****")


# Flag duplicate NPI numbers
silver_df['NPI_dupe'] = silver_df['NPI'].duplicated()

# Flag duplicate practice address per physician
silver_df['address_dupe'] = silver_df[['First_Name','Last_Name','Practice_Address_line_1','Zip']].duplicated()

# Flag duplciate phone number per physician
silver_df['phone_dupe'] = silver_df[['First_Name','Last_Name','Phone_std']].duplicated()


# Score the uniqueness of each record by deducting points for dupes
# CURRENTLY DEDUCTING 10 POINTS FOR EACH DUPE BUT COULD CHANGE BASED ON DUPE WEIGHTING
silver_df['uniqueness_score'] = 100 
silver_df.loc[silver_df['NPI_dupe'] == True, 'uniqueness_score'] -= 10
silver_df.loc[silver_df['address_dupe'] == True, 'uniqueness_score'] -= 10
silver_df.loc[silver_df['phone_dupe'] == True, 'uniqueness_score'] -= 10


print(f"Average Uniqueness Score: {silver_df['uniqueness_score'].mean().round(2)}")

**DQS Calculation**

Since Uniqueness and Completeness both are Equally important to generate Golden Records, we shall assign 50% to each score

In [0]:
silver_df['DQ_Score'] = (silver_df['completeness_score'] + silver_df['uniqueness_score']) * 0.50

silver_df['DQ_Score'].describe()


In [0]:
silver_df.groupby('Source_System')['DQ_Score'].describe()['mean']

# Golden Record Creation

First, look into the unique NPIs and Name combinations to get a sense of the cleanup needed

In [0]:
from rapidfuzz import fuzz, process

# ==== Fuzzy Matching ====

silver_df['full_name'] = silver_df['First_Name'] + ' ' + silver_df['Last_Name']

unique_full_names = silver_df['full_name'].dropna().unique().tolist()
print(f"Out of {len(silver_df)} names, total unique names: {len(unique_full_names)} and \n unique NPIs: {silver_df['NPI'].dropna().nunique()}")

matches = []

for name in unique_full_names:
    # get all records with the same name
    match_scores = process.extract(name, unique_full_names, scorer=fuzz.token_sort_ratio, limit=100)
    
    for match, score, ind in match_scores:
        if score > 80 and name != match:    #exclude the matches with the same name
            matches.append({
                'Name_1': name,
                'Name_2': match,
                'Score': round(score, 2)
            })

# Add all the duplicate names to a dataframe           
matches_df = pd.DataFrame(matches).drop_duplicates()
print(f"total likely duplicated names: {matches_df.shape[0]} \n the score distribution:\n{matches_df['Score'].describe().round(2)}")


        


In [0]:
matches_df['Score'].hist(bins=100)
print(len(matches_df[matches_df['Score'] > 93]))

In [0]:
# Categorize duplicate records based on match score

confidence_levels = {
    'High': 93,
    'Medium': 88,
    'Low': 70
}

def categorize_score(score):
    for level, threshold in confidence_levels.items():
        if score >= threshold:
            return level
    return 'Unknown'

matches_df['Confidence_level'] = matches_df['Score'].apply(categorize_score)
# Get the number of records with each confidence level
print(f"confidence level counts:  {matches_df['Confidence_level'].value_counts()}")


In [0]:
matches_df.head(5)

In [0]:
silver_name_match_df = silver_df.merge(matches_df[['Name_1', 'Name_2', 'Confidence_level']], left_on='full_name', right_on='Name_1', how='left').drop(['Name_1'], axis=1)

In [0]:
silver_name_match_df.head(5)

# Survivorship Rules

Rule 1 - Most Frequently occuring names get preference

Rule 2 - In the event the occurrence counts are the same, prefer the one with greater length. However, this rule would not work for records when there is a typo with the same length of the name. That would be left for manual review.

In [0]:
# function to pick the survivor name

def survivor_name(name_1, name_2, df):
    name_1_cnt = df[df['full_name'] == name_1].shape[0]
    name_2_cnt = df[df['full_name'] == name_2].shape[0]
    
    if name_1_cnt > name_2_cnt:
        return name_1
    elif name_1_cnt == name_2_cnt:
        if len(name_1) > len(name_2):
            return name_1
        elif len(name_1) == len(name_2):
            return "Need manual review"
        return name_2
    return name_2 

In [0]:
silver_name_match_df.head()

In [0]:
golden_df = silver_name_match_df.copy()



Apply Survivor names

In [0]:
import numpy as np

golden_df['golden_name'] = np.where(
    golden_df['Confidence_level'].isna(),
    golden_df['full_name'],
    np.where(
        golden_df['Confidence_level'] == 'High',
        golden_df.apply(
            lambda x: survivor_name(
                name_1=x['full_name'],
                name_2=x['Name_2'],
                df=golden_df
            ),
            axis=1
        ),
        "Need manual review"
    )
)

golden_df.head()

In [0]:
golden_df['Source_System'].value_counts()


In [0]:
for loser, survivor in tracking_matches.items():
    golden_df.loc[golden_df['full_name'] == loser, 'golden_name'] = survivor

Practice Address Explore

In [0]:
# check how many addresses are associated with each NPI
add_per_npi = silver_df.groupby('NPI')['Practice_Address_line_1'].nunique()
multi_addr = add_per_npi[add_per_npi > 1]

# explore the condition of the NPIs with multiple addresses
multi_addr_npi = multi_addr.index.to_list()
multi_addr_npi

In [0]:
silver_df[silver_df['Practice_Address'].isna()]


The following code takes care of addresses that have NPI values but for one occurances of the NPI, there is an address while the other occurance does not. 

Next need to take care of the cases where the name has a match/fuzzy match but one of the occurances is missing the NPI. Hence, the address needs to match.

Next, take care of addresses where the name has no address at all. 

In [0]:
multi_addr_df = silver_df[silver_df['NPI'].isin(multi_addr_npi)]

empty_mask = multi_addr_df.applymap(lambda x: isinstance(x,str))
empty_mask
# all missing multi address NPIs have original addresses in their counterparts
# All empty cells in Practice_address field are None

In [0]:
multi_addr_df.dropna(subset=['Practice_Address'], inplace=True)
multi_addr_df